# XGBOOST Model - Analysis

In [1]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.model_selection import RandomizedSearchCV

In [2]:
# ─────────────────────────────────────────
# 1. CHARGEMENT DES DONNÉES
# ─────────────────────────────────────────
train_df = pd.read_csv('../data/data_prep_1/train_1.csv')
val_df   = pd.read_csv('../data/data_prep_1/val_1.csv')
test_df  = pd.read_csv('../data/data_prep_1/test_1.csv')

target = 'target_is_fraud'

In [3]:
test_df.shape

(40000, 22)

In [21]:
# 2. SÉPARATION X (FEATURES) ET Y (TARGET)
# On sépare les features de la target pour entraîner le modèle
# Le customer_id doit être retiré car ce n'est pas une feature prédictive
# ═════════════════════════════════════════════════════════════════════════════

# TRAIN : pas de customer_id (SMOTE a généré des lignes synthétiques sans ID)
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

# VAL : retirer customer_id si présent
if 'customer_id' in val_df.columns:
    X_val = val_df.drop(columns=[target, 'customer_id'])
else:
    X_val = val_df.drop(columns=[target])
y_val = val_df[target]

# TEST : garder customer_id séparément pour la submission finale
customer_ids = test_df['customer_id']
X_test = test_df.drop(columns=['customer_id'])

In [23]:
param_grid = {
    'n_estimators': [50, 100, 150],          # ← RÉDUIT (moins d'arbres)
    'max_depth': [2, 3, 4],                  # ← TRÈS BAS (arbres peu profonds)
    'learning_rate': [0.01, 0.05],           # ← PLUS LENT (apprend moins vite)
    'subsample': [0.5, 0.6, 0.7],            # ← BAS (50-70% des lignes)
    'colsample_bytree': [0.5, 0.6, 0.7],     # ← BAS (50-70% des features)
    'min_child_weight': [5, 7, 10],          # ← TRÈS HAUT (feuilles lourdes)
    'gamma': [0.3, 0.5, 0.7],                # ← TRÈS HAUT (pénalise les splits)
    'reg_alpha': [1, 2, 3],                  # ← TRÈS HAUT (L1 fort)
    'reg_lambda': [3, 5, 7]                  # ← TRÈS HAUT (L2 fort)
}

In [24]:
# 5. CRÉATION DU MODÈLE DE BASE
# On définit les paramètres fixes qui ne seront pas tunés
# ═════════════════════════════════════════════════════════════════════════════
base_model = xgb.XGBClassifier(
    random_state=42,
    eval_metric='aucpr',
    tree_method='hist',                      # accélère l'entraînement
    n_jobs=-1                                # utilise tous les CPU
)

In [25]:
# ═════════════════════════════════════════════════════════════════════════════
# ÉTAPE 5 : RANDOMIZED SEARCH POUR TROUVER LES MEILLEURS HYPERPARAMÈTRES
# n_iter=15 : teste 15 combinaisons aléatoires
# cv=2 : validation croisée 2-fold (balance vitesse/fiabilité)
# scoring='f1' : optimise le F1-Score
# ═════════════════════════════════════════════════════════════════════════════
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_grid,
    n_iter=50,
    cv=2,
    scoring='f1',
    verbose=2,
    n_jobs=-1,
    random_state=42
)

print("\n" + "="*70)
print("DÉBUT DU TUNING (estimation: 15-20 min)")
print("="*70 + "\n")

random_search.fit(X_train, y_train)


DÉBUT DU TUNING (estimation: 15-20 min)

Fitting 2 folds for each of 50 candidates, totalling 100 fits


,estimator,"XGBClassifier...ree=None, ...)"
,param_distributions,"{'colsample_bytree': [0.5, 0.6, ...], 'gamma': [0.3, 0.5, ...], 'learning_rate': [0.01, 0.05], 'max_depth': [2, 3, ...], ...}"
,n_iter,50
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,2
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [28]:
# ÉTAPE 6 : AFFICHAGE DES MEILLEURS HYPERPARAMÈTRES TROUVÉS
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("MEILLEURS PARAMÈTRES TROUVÉS :")
print("="*70)
for param, value in random_search.best_params_.items():
    print(f"{param:20s} : {value}")

print("\n" + "="*70)
print(f"MEILLEUR F1-SCORE (train avec CV) : {random_search.best_score_:.4f}")
print("="*70)

best_model = random_search.best_estimator_


MEILLEURS PARAMÈTRES TROUVÉS :
subsample            : 0.6
reg_lambda           : 3
reg_alpha            : 2
n_estimators         : 150
min_child_weight     : 7
max_depth            : 4
learning_rate        : 0.05
gamma                : 0.3
colsample_bytree     : 0.6

MEILLEUR F1-SCORE (train avec CV) : 0.8635


In [9]:
y_train_pred = best_model.predict(X_train)
y_val_pred = best_model.predict(X_val)

f1_train = f1_score(y_train, y_train_pred)
f1_val = f1_score(y_val, y_val_pred)

print("\n" + "="*70)
print("DÉTECTION D'OVERFIT :")
print("="*70)
print(f"F1-Score TRAIN : {f1_train:.4f}")
print(f"F1-Score VAL   : {f1_val:.4f}")
print(f"Écart          : {f1_train - f1_val:.4f}")

if f1_train - f1_val > 0.15:
    print("⚠️  OVERFIT DÉTECTÉ ! Écart > 0.15")
elif f1_train - f1_val > 0.10:
    print("⚠️  Léger overfit (écart entre 0.10 et 0.15)")
else:
    print("✅ Pas d'overfit majeur (écart < 0.10)")


DÉTECTION D'OVERFIT :
F1-Score TRAIN : 0.8694
F1-Score VAL   : 0.4073
Écart          : 0.4621
⚠️  OVERFIT DÉTECTÉ ! Écart > 0.15


In [10]:
# ÉTAPE 8 : OPTIMISATION DU SEUIL DE DÉCISION
# Le seuil par défaut (0.5) n'est pas optimal pour la fraude
# On teste différents seuils pour maximiser le F1-Score
# Seuil bas → détecte plus de fraudes (recall ↑) mais plus de faux positifs
# Seuil haut → moins de faux positifs (precision ↑) mais rate des fraudes
# ═════════════════════════════════════════════════════════════════════════════
y_val_proba = best_model.predict_proba(X_val)[:, 1]

print("\n" + "="*70)
print("OPTIMISATION DU SEUIL DE DÉCISION :")
print("="*70)
print(f"{'Seuil':<8} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
print("-"*70)

best_f1 = 0
best_threshold = 0.5

for threshold in np.arange(0.2, 0.6, 0.05):
    y_pred_thresh = (y_val_proba >= threshold).astype(int)
    
    precision = precision_score(y_val, y_pred_thresh)
    recall = recall_score(y_val, y_pred_thresh)
    f1 = f1_score(y_val, y_pred_thresh)
    
    print(f"{threshold:<8.2f} {precision:<12.4f} {recall:<12.4f} {f1:<12.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print("-"*70)
print(f"🎯 MEILLEUR SEUIL : {best_threshold:.2f} avec F1 = {best_f1:.4f}")
print("="*70)


OPTIMISATION DU SEUIL DE DÉCISION :
Seuil    Precision    Recall       F1-Score    
----------------------------------------------------------------------
0.20     0.1540       0.9198       0.2639      
0.25     0.1739       0.8824       0.2906      
0.30     0.1954       0.8454       0.3174      
0.35     0.2174       0.8015       0.3421      
0.40     0.2417       0.7521       0.3659      
0.45     0.2714       0.7005       0.3912      
0.50     0.2990       0.6385       0.4073      
0.55     0.3268       0.5590       0.4125      
----------------------------------------------------------------------
🎯 MEILLEUR SEUIL : 0.55 avec F1 = 0.4125


In [11]:
# ÉTAPE 9 : ÉVALUATION FINALE SUR VAL AVEC LE MEILLEUR SEUIL
# ═════════════════════════════════════════════════════════════════════════════
y_val_pred_final = (y_val_proba >= best_threshold).astype(int)

print("\n" + "="*70)
print(f"ÉVALUATION FINALE SUR VAL (seuil = {best_threshold:.2f}) :")
print("="*70)
print(f"Accuracy  : {accuracy_score(y_val, y_val_pred_final):.4f}")
print(f"F1-Score  : {f1_score(y_val, y_val_pred_final):.4f}")
print(f"Precision : {precision_score(y_val, y_val_pred_final):.4f}")
print(f"Recall    : {recall_score(y_val, y_val_pred_final):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_val, y_val_proba):.4f}")
print("\nRapport de classification :")
print(classification_report(y_val, y_val_pred_final))


ÉVALUATION FINALE SUR VAL (seuil = 0.55) :
Accuracy  : 0.8629
F1-Score  : 0.4125
Precision : 0.3268
Recall    : 0.5590
ROC-AUC   : 0.8437

Rapport de classification :
              precision    recall  f1-score   support

           0       0.96      0.89      0.92     29245
           1       0.33      0.56      0.41      2755

    accuracy                           0.86     32000
   macro avg       0.64      0.73      0.67     32000
weighted avg       0.90      0.86      0.88     32000



In [12]:
# ÉTAPE 10 : PRÉDICTIONS FINALES SUR TEST AVEC LE MEILLEUR SEUIL
# On applique le seuil optimisé trouvé sur val pour prédire sur test
# ═════════════════════════════════════════════════════════════════════════════
test_pred_proba = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_pred_proba >= best_threshold).astype(int)

submission = pd.DataFrame({
    'customer_id': customer_ids,
    'target_is_fraud': test_pred
})

submission.to_csv('submission_test.csv', index=False)

print("\n" + "="*70)
print("SUBMISSION CRÉÉE : submission_optimized.csv")
print("="*70)
print(submission.head(10))
print(f"\nNombre total de prédictions : {len(submission)}")
print(f"Fraudes détectées : {test_pred.sum()} ({test_pred.sum()/len(test_pred)*100:.2f}%)")
print("="*70)


SUBMISSION CRÉÉE : submission_optimized.csv
       customer_id  target_is_fraud
0  CUST_E5RX1BC9II                0
1  CUST_BHWIUKERGN                1
2  CUST_EXT9NA4CHU                0
3  CUST_9FSJE5R1NY                0
4  CUST_GDQXMODBED                0
5  CUST_8S20TEQETL                1
6  CUST_MEJIHBWAMV                0
7  CUST_ZVLC1MRRAF                1
8  CUST_WP2LSISPLM                0
9  CUST_UFMY9N6P78                1

Nombre total de prédictions : 40000
Fraudes détectées : 5737 (14.34%)


## Minimisation du cout du modele

In [16]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.metrics import (accuracy_score, classification_report, roc_auc_score, 
                             f1_score, precision_score, recall_score, confusion_matrix, make_scorer)
from sklearn.model_selection import RandomizedSearchCV

In [17]:
# ═════════════════════════════════════════════════════════════════════════════
# 3. SCORER MÉTIER PERSONNALISÉ
# Minimise FN × 150 + FP × 10 (retourné en négatif car sklearn maximise)
# ═════════════════════════════════════════════════════════════════════════════
cost_fn = 150  # fraude non détectée
cost_fp = 10   # client bloqué à tort

def cout_metier(y_true, y_pred, cost_fn=cost_fn, cost_fp=cost_fp):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return -(fn * cost_fn + fp * cost_fp)

scorer_metier = make_scorer(cout_metier)

In [18]:
# 4. GRILLE D'HYPERPARAMÈTRES
# ═════════════════════════════════════════════════════════════════════════════
param_grid = {
    'n_estimators':      [50, 100, 150],
    'max_depth':         [2, 3, 4],
    'learning_rate':     [0.01, 0.05],
    'subsample':         [0.5, 0.6, 0.7],
    'colsample_bytree':  [0.5, 0.6, 0.7],
    'min_child_weight':  [5, 7, 10],
    'gamma':             [0.3, 0.5, 0.7],
    'reg_alpha':         [1, 2, 3],
    'reg_lambda':        [3, 5, 7]
}

In [19]:
# 5. MODÈLE DE BASE
# ═════════════════════════════════════════════════════════════════════════════
base_model = xgb.XGBClassifier(
    random_state=42,
    eval_metric='aucpr',
    tree_method='hist',
    n_jobs=-1
)

# ═════════════════════════════════════════════════════════════════════════════
# 6. RANDOMIZED SEARCH AVEC SCORER MÉTIER
# ═════════════════════════════════════════════════════════════════════════════
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_grid,
    n_iter=50,
    cv=2,
    scoring=scorer_metier,  # optimisation métier, pas F1
    verbose=2,
    n_jobs=-1,
    random_state=42
)
print("\n" + "="*70)
print("DÉBUT DU TUNING - Optimisation du coût métier (FN×150 + FP×10)")
print("="*70 + "\n")

random_search.fit(X_train, y_train)

print("\n" + "="*70)
print("MEILLEURS PARAMÈTRES TROUVÉS :")
print("="*70)
for param, value in random_search.best_params_.items():
    print(f"{param:20s} : {value}")
print(f"\nMEILLEUR SCORE MÉTIER (CV) : {-random_search.best_score_:,.0f}€")
print("="*70)

best_model = random_search.best_estimator_


DÉBUT DU TUNING - Optimisation du coût métier (FN×150 + FP×10)

Fitting 2 folds for each of 15 candidates, totalling 30 fits

MEILLEURS PARAMÈTRES TROUVÉS :
subsample            : 0.6
reg_lambda           : 3
reg_alpha            : 2
n_estimators         : 150
min_child_weight     : 7
max_depth            : 4
learning_rate        : 0.05
gamma                : 0.3
colsample_bytree     : 0.6

MEILLEUR SCORE MÉTIER (CV) : 1,270,675€


In [20]:
# 8. OPTIMISATION DU SEUIL PAR COÛT MÉTIER
# ═════════════════════════════════════════════════════════════════════════════
y_val_proba = best_model.predict_proba(X_val)[:, 1]

print("\n" + "="*70)
print("OPTIMISATION DU SEUIL DE DÉCISION (coût métier) :")
print("="*70)
print(f"{'Seuil':<8} {'FN':<8} {'FP':<8} {'Coût Total (€)':<18} {'F1-Score':<10}")
print("-"*55)

best_cost = float('inf')
best_threshold_cost = 0.5

for threshold in np.arange(0.20, 0.60, 0.05):
    y_pred_thresh = (y_val_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_thresh).ravel()
    f1 = f1_score(y_val, y_pred_thresh)
    total_cost = fn * cost_fn + fp * cost_fp
    print(f"{threshold:<8.2f} {fn:<8} {fp:<8} {total_cost:<18,} {f1:<10.4f}")
    if total_cost < best_cost:
        best_cost = total_cost
        best_threshold_cost = threshold

print("-"*55)
print(f"💰 SEUIL OPTIMAL (coût métier) : {best_threshold_cost:.2f} | Coût minimal : {best_cost:,}€")
print("="*70)


OPTIMISATION DU SEUIL DE DÉCISION (coût métier) :
Seuil    FN       FP       Coût Total (€)     F1-Score  
-------------------------------------------------------
0.20     221      13917    172,320            0.2639    
0.25     324      11546    164,060            0.2906    
0.30     426      9591     159,810            0.3174    
0.35     547      7947     161,520            0.3421    
0.40     683      6499     167,440            0.3659    
0.45     825      5182     175,570            0.3912    
0.50     996      4123     190,630            0.4073    
0.55     1215     3172     213,970            0.4125    
-------------------------------------------------------
💰 SEUIL OPTIMAL (coût métier) : 0.30 | Coût minimal : 159,810€
